#### Predictive Modelling for Multiphase Flow Regime Classification using High-Frequency Pressure Sensors

##### Part 1: Extraction, Transformation and Storage of the PRONTO benchmark dataset

##### **Purpose**: This notebook performs the calibration and windowing of high-frequency pressure sensor data to generate a well-structured dataset for flow regime classification.

##### **Author:** Bello Oluwatobi

##### **Last Updated:** November 22, 2025

### #1 Importing Libraries

In [ ]:
# importing necessary libraries
import pandas as pd
import numpy as np
import glob
import os
import re

### #2 Parameters setup & loading the raw files

In [ ]:
# setting up the parameters
PIPE_DIAMETER_METERS = 0.0508  # PRONTO rig is 2 inches (50.8mm)
PIPE_AREA = np.pi * (PIPE_DIAMETER_METERS / 2)**2  # Area in m²

In [ ]:
# defining the paths to the data directories
directories = {
    0: '../data/Test9_high_frequency_pressure/*.csv',   # label 0: Normal condition
    1: '../data/Test10_high_frequency_pressure/*.csv'  # Label 1: Slugging condition
}

# displaying the pipe area for reference
print(f"Pipe Area (Reference): {PIPE_AREA:.6f} m²")

### #3 Extracting and processing the raw data

In [ ]:
# initializing an empty list to hold all processed data
dataset = []

# configuring the sliding window parameters
SAMPLING_RATE = 5000 
WINDOW_DURATION = 0.25 
CHUNK_SIZE = int(SAMPLING_RATE * WINDOW_DURATION)

In [ ]:
# performing data extraction and processing

# looping through each directory and processing files
for label, path in directories.items():
    files = glob.glob(path)
    print(f"Processing Class {label}: {len(files)} files found.")
    
    # Loop through each file in the directory
    for file_path in files:
        # opening the file to read header lines
        with open(file_path, 'r', encoding='latin-1') as f:
            header_lines = [f.readline().strip() for _ in range(20)]
        
        header_row_idx = 0
        for i, line in enumerate(header_lines):
            if "Time" in line and "P4" in line:
                header_row_idx = i
                break
        
        # loading the CSV data, skipping the header rows
        df = pd.read_csv(file_path, sep=None, engine='python', encoding='latin-1', 
                            header=header_row_idx, skiprows=[header_row_idx+1, header_row_idx+2])
        df = df.dropna(axis=1, how='all')
        df.columns = [c.strip() for c in df.columns]
        
        # extracting flow parameters from the filename
        filename = os.path.basename(file_path)
        parts = filename.replace('.csv', '').split('_')
        
        # setting default flow values
        air_flow_sm3h = 0.0
        water_flow_kgs = 0.0
        
        # mapping the filename shorthands to actual values
        if len(parts) >= 3:
            air_flow_sm3h = float(parts[-2])
            w_str = parts[-1]
        
            if w_str == "01": water_flow_kgs = 0.1
            elif w_str == "05": water_flow_kgs = 0.5
            elif w_str == "1": water_flow_kgs = 1.0
            elif w_str == "2": water_flow_kgs = 2.0
            elif w_str == "35": water_flow_kgs = 3.5
            else: water_flow_kgs = float(w_str)

        # processing data in 250ms windows
        num_chunks = len(df) // CHUNK_SIZE
        target_sensors = ['P4', 'p5', 'P6', 'P7', 'P8', 'P13', 'P14', 'P15', 'P17']
        
        for i in range(num_chunks):
            start = i * CHUNK_SIZE
            end = start + CHUNK_SIZE
            window = df.iloc[start:end]
            
            # setting up the data row with flow parameters and label
            row = {
                'Air_Flow_sm3h': air_flow_sm3h,
                'Water_Flow_kgs': water_flow_kgs,
                'Pipe_Diameter_m': PIPE_DIAMETER_METERS,
                'Target_Label': label
            }
            
            # calculating mean and std for each pressure sensor
            for col in window.columns:
                if col in target_sensors:
                    # converting to numeric values and handling errors
                    vals = pd.to_numeric(window[col], errors='coerce').dropna()
                    if not vals.empty:
                        row[f'{col}_Mean'] = vals.mean()
                        row[f'{col}_Std'] = vals.std()
            
            dataset.append(row)

In [ ]:
# creating the dataframe
final_df = pd.DataFrame(dataset)

# displaying the total number of samples extracted
print(f"Total Samples: {len(final_df)}")

In [ ]:
# converting air flowrate: (sm3/h) to (m3/s) approx
final_df['Q_gas_m3s'] = final_df['Air_Flow_sm3h'] / 3600.0

# converting water flowrate: (kg/s) to (m3/s) - density of water approx 1000 kg/m3
final_df['Q_liquid_m3s'] = final_df['Water_Flow_kgs'] / 1000.0

# calculatig the superficial velocities (Vsg, Vsl)
final_df['Vsg'] = final_df['Q_gas_m3s'] / PIPE_AREA
final_df['Vsl'] = final_df['Q_liquid_m3s'] / PIPE_AREA

### #4 Storing the processed data

In [ ]:
# storing the processed data to a CSV file
final_df.to_csv('../data/processed_data.csv', index=False)

final_df.head()